In [14]:
%matplotlib widget
import itertools

import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import polars as pl
import pybamm
import tqdm.auto as tqdm
from scipy.integrate import cumulative_trapezoid
from scipy.optimize import curve_fit

from utils import fit_polarization

In [15]:
def decay(t, OCV: float, U0: float, log_tau: float):
    tau = 10**log_tau
    return OCV + U0 * np.exp(-t / tau)


def fit_decay(t, V, sigma=1e-3):

    if V[-1] > V[0]:
        lo, hi = [V[-1], -1, 2], [4.2, 0, 36000]
    elif V[-1] < V[0]:
        lo, hi = [2.5, 0, 2], [V[-1], 1, 36000]
    else:
        lo, hi = [2.5, -1, 2], [4.2, 1, 36000]
    p0 = [V[-1], 0.0, 100]

    popt, pcov = curve_fit(
        decay,
        t,
        V,
        p0=p0,
        sigma=sigma,
        absolute_sigma=True,
        # bounds=(lo, hi),
        maxfev=1000000,
        # jac="3-point",
    )
    perr = np.sqrt(np.diag(pcov))
    results = {
        "Open-circuit voltage / V": popt[0],
        "Element-1 initial overpotential / V": popt[1],
        "Tau1 / s": popt[2],
        "Open-circuit voltage stdev / V": perr[0],
        "Element-1 initial overpotential stdev / V": perr[1],
        "Tau1 stdev / s": perr[2],
    }
    return results

In [16]:
def run(c_rate_: float, rest_time: int):
    model = pybamm.lithium_ion.DFN()
    param = pybamm.ParameterValues("OKane2022")
    param.update(
        {
            "Upper voltage cut-off [V]": 4.5,
            "Lower voltage cut-off [V]": 2.0,
        }
    )
    pulse_dod = 0.005
    pulse_time = pulse_dod / c_rate_ * 3600  # seconds
    n_pulses = int(1 / pulse_dod)  # integer division
    expr_ = []
    for i in range(n_pulses):
        expr_.append(
            (
                f"Charge at {c_rate_}C for {pulse_time} seconds",
                f"Rest for {rest_time} seconds",
            )
        )

    experiment = pybamm.Experiment(
        expr_,  # type:ignore
        period="1 second",
    )
    simulation = pybamm.Simulation(model, parameter_values=param, experiment=experiment)
    solution = simulation.solve(initial_soc=0.0)

    results = {
        "Mode / 1": [],
        "Time / s": [],
        "Current / A": [],
        "Voltage / V": [],
        "Step Count / 1": [],
    }
    step_count = 1
    for cycle in solution.cycles:
        for mode, step in zip(["Charge", "Rest"], cycle.sub_solutions):
            results["Mode / 1"].extend([mode] * len(step["Time [s]"].entries))
            results["Time / s"].extend(step["Time [s]"].entries)
            results["Current / A"].extend(step["Current [A]"].entries)
            results["Voltage / V"].extend(step["Terminal voltage [V]"].entries)
            results["Step Count / 1"].extend(
                [step_count] * len(step["Time [s]"].entries)
            )
            step_count += 1
    results = (
        pl.DataFrame(results)
        .with_columns(
            (
                (
                    -(1 / 3600)
                    * pl.col("Time / s").diff()
                    * pl.col("Current / A").shift(1).cum_sum()
                ).fill_null(0)
            ).alias("Charge / Ah")
        )
        .with_columns(
            (
                (pl.col("Charge / Ah") - pl.col("Charge / Ah").min())
                / (pl.col("Charge / Ah").max() - pl.col("Charge / Ah").min())
            ).alias("State of Charge / 1")
        )
    )
    return pl.DataFrame(results)


def decay(t: np.ndarray, OCV: float, U: float, Tau: float):
    return OCV + U * np.exp(-t / Tau)


def fit(results: pl.DataFrame):
    parameters = []
    for step in results.filter(pl.col("Mode / 1") == "Rest").partition_by(
        "Step Count / 1"
    ):
        t, V, SOC = (
            step.select(["Time / s", "Voltage / V", "State of Charge / 1"]).to_numpy().T
        )
        parameters.append({**fit_decay(t - t[0], V), "State of Charge / 1": SOC.mean()})
    return pl.DataFrame(parameters).sort("State of Charge / 1")


parameters = []
combos = list(itertools.product([0.05, 0.1, 0.5, 1.0], [10, 60, 300, 600, 1800]))
for c_rate, rest_time in tqdm.tqdm(combos):
    sol = run(c_rate, rest_time)
    parameters.append(
        fit(sol).with_columns(
            pl.lit(c_rate).alias("C-rate / 1"), pl.lit(rest_time).alias("Rest time / s")
        )
    )

parameters = pl.concat(parameters).sort(
    ["C-rate / 1", "Rest time / s", "State of Charge / 1"]
)

  0%|          | 0/20 [00:00<?, ?it/s]

In [22]:
fig = px.scatter(
    parameters,
    x="State of Charge / 1",
    y="Open-circuit voltage / V",
    facet_col="C-rate / 1",
    color="Rest time / s",
)
fig.show()

fig = px.scatter(
    parameters,
    x="State of Charge / 1",
    y="Open-circuit voltage stdev / V",
    facet_col="C-rate / 1",
    color="Rest time / s",
)
fig.show()

fig = px.scatter(
    parameters,
    x="State of Charge / 1",
    y="Tau1 / s",
    facet_col="C-rate / 1",
    color="Rest time / s",
)
fig.show()

fig = px.scatter(
    parameters,
    x="State of Charge / 1",
    y="Tau1 stdev / s",
    facet_col="C-rate / 1",
    color="Rest time / s",
)
fig.show()
fig = px.scatter(
    parameters,
    x="Rest time / s",
    y="Open-circuit voltage / V",
    facet_col="C-rate / 1",
    color="State of Charge / 1",
)
fig.show()
fig = px.scatter(
    parameters,
    x="Rest time / s",
    y="Tau1 / s",
    facet_col="C-rate / 1",
    color="State of Charge / 1",
)
fig.show()

In [18]:
sol = run(0.05, 10)
px.line(sol, x="Time / s", y="Voltage / V")